<a href="https://colab.research.google.com/github/TienNguyen0712/hybrid-llm-tabular-pipeline-for-icu-mortality-prediction/blob/main/mimic_iv_rag_pipeline_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trích xuất thông tin & So sánh hồ sơ tương đồng từ bộ MIMIC-IV dể giải quyết bài toán Dự đoán mức độ sinh tồn**

## **Tổng quan**

Notebook được viết nhằm thực nghiệm các bước triển khai của pipeline được cung cấp


In [2]:
# ── CELL 1: Cài thư viện + Mount Drive ──────────────────────
# Bỏ comment nếu chạy trên Google Colab
from google.colab import drive
drive.mount('/content/drive')
!pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

from typing import List, Tuple, Dict

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
✓ Libraries loaded


## **Cấu hình đường dẫn**

In [3]:
# ── CELL 2: Đường dẫn dữ liệu ───────────────────────────────
# === CHỈNH Ở ĐÂY ===
# DATA_DIR = r"E:\KLTN\mimiciv\3.1\parquet"  # Windows local
DATA_DIR = "/content/drive/MyDrive/NCKH-DDU1231/mimic-iv-clinical-database-demo-2.2/mimic-iv-clinical-database-demo-2.2"  # Google Colab

import os
def load(folder, name):
    path = os.path.join(DATA_DIR, folder, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY")
        return pd.DataFrame()

## **Nạp dữ liệu cơ bản**

### **Xây dựng bảng cohort dựa theo các điều kiện**

Xây dựng một bảng cohort được gộp từ bảng `patients`, `admissions`, `icustay`. Điều kiện gộp và lọc bao gồm:
- Các bệnh nhân phải là người trưởng thành (trên 18 tuổi)
- Lấy đợt năm ICU dầu tiên
- Thời gian nằm viện phải trên 24h kể từ khi vào ICU

Lựa chọn biến mục tiêu là: `hospital_expire_flag` (Tử vong trong đợt nằm viện)


In [4]:
def build_and_val_icu_cohort(patients_df: pd.DataFrame, admissions_df: pd.DataFrame, icustays_df: pd.DataFrame) -> pd.DataFrame:
  # Chuyển đổi định dạng mốc thời gian sang Datetime
  time_cols = {
      'icustays_df': ['intime', 'outtime'],
      'admissions_df': ['admittime', 'dischtime', 'edregtime', 'edouttime'],
      'patients_df' : []
  }

  for col in time_cols['icustays_df']:
        icustays_df[col] = pd.to_datetime(icustays_df[col])
  for col in time_cols['admissions_df']:
        admissions_df[col] = pd.to_datetime(admissions_df[col])
  # 1. Inner Join với patients
  cohort = icustays_df.merge(
  admissions_df[["subject_id", "hadm_id", "admittime", "dischtime", "admission_type",
                        "admission_location", "edregtime", "edouttime", "deathtime",
                        "hospital_expire_flag", "insurance", "race"]],
          on=['subject_id', 'hadm_id'],
          how='inner'
      )
  # 2. Inner join với admission
  cohort = cohort.merge(
          patients_df[["subject_id", "gender", "anchor_age", "anchor_year"]],
          on='subject_id',
          how='inner'
      )

  initial_count = len(cohort)
  print(f"Tổng số lượt ICU ban đầu: {initial_count:,}")

  # Áp dụng các tiêu chí lọc
  # ---------------------------------------------
  # Tiêu chí 1: Người trưởng thành (>= 18 tuổi)
  # Có thể thực hiện cách khác khi lọc người trường thành:
  """
  Tuổi = anchor_age + (năm của icustays.intime - anchor_y)
  """
  # ----------------------------------------------
  cohort = cohort[cohort["anchor_age"] >= 18]
  print(f"-> Sau khi lọc người trưởng thành (>=18t): {len(cohort):,} ca")

  # ---------------------------------------------
  # Tiêu chí 2: Lấy ca ICU đầu tiên của mỗi bệnh nhân (First ICU stay per patient)
  # Sắp xếp theo intime để chắc chắn lấy ca đầu tiên trong đời/lịch sử của bệnh nhân
  # ----------------------------------------------
  cohort = (
      cohort.sort_values(["subject_id", "intime"])
            .groupby("subject_id", as_index=False)
            .first()
  )

  print(f"-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: {len(cohort):,} ca")

  # Tính thời gian nằm ICU (tính theo ngày)
  cohort["icu_los_days"] = (
      pd.to_datetime(cohort["outtime"]) - pd.to_datetime(cohort["intime"])
  ).dt.total_seconds() / 86400
  cohort["icu_los_hours"] = cohort["icu_los_days"] * 24

  # ---------------------------------------------
  # Tiêu chí 3: ICU stay trên 1 ngày (> 1 ngày, tức > 24 giờ)
  # ---------------------------------------------
  cohort = cohort[cohort["icu_los_days"] > 1.0]
  print(f"-> Sau khi lọc ICU stay > 1 ngày: {len(cohort):,} ca")

  # Tạo mốc thời gian kết thúc cửa sổ quan sát (intime + 24h)
  cohort['obs_end_time'] = cohort['intime'] + pd.Timedelta(hours=24)

  # Xây dựng nhóm tuổi
  cohort["age_group"] = pd.cut(
      cohort["anchor_age"],
      bins=[17, 30, 50, 65, 80, 120],
      labels=["18-30", "31-50", "51-65", "66-80", "80+"]
  )

  print("=== BẮT ĐẦU THỰC HIỆN Kiểm tra logic ===")
  """
  Tiêu chí loại bỏ các lượt ICU vi phạm logic:
  - Tiêu chí 1: Các khóa phải là duy nhất
  - Tiêu chí 2: Kiểm tra thứ tự các môc thời gian logic
    - Thời gian nhập voeekm <= Thời gian vào ICU < Thời gian vào ICU + 24h <= Thời gian ra ICU <= Thời gian ra viện
  - Tiêu chí 3: Kiểm tra tính hợp lệ của Nhãn Mục tiêu
    - Thời điểm tử vong của bệnh nhân phải lớn hơn thười gian vào ICU + 24h

  """

  # List lưu giữ các stay_id vi phạm cần loại bỏ
  invalid_stay_ids = set()

  # Check 1: Kiểm tra tính duy nhất của Khóa
  total_rows = len(cohort)
  unique_stays = cohort['stay_id'].nunique()
  print(f"Tổng số dòng: {total_rows} | Số stay_id duy nhất: {unique_stays}")
  if total_rows != unique_stays:
      print("Bị trùng lặp stay_id do phép Join! Cần loại bỏ bản ghi trùng.")
      cohort = cohort.drop_duplicates(subset=['stay_id'])

  # Check 2: Kiểm tra thứ tự mốc thời gian logic
  invalid_time_mask = (
        (cohort['admittime'] > cohort['intime']) |
        (cohort['intime'] >= cohort['outtime']) |
        (cohort['outtime'] > cohort['dischtime'])
    )
  time_faulty_ids = cohort[invalid_time_mask]['stay_id'].tolist()
  invalid_stay_ids.update(time_faulty_ids)
  print(f"Phát hiện {len(time_faulty_ids)} lượt ICU vi phạm thứ tự thời gian sinh lý.")

  # Check 3: Kiểm tra tính hợp lệ của Nhãn Mục tiêu
  early_death_mask = (
        (cohort['hospital_expire_flag'] == 1) &
        (cohort['deathtime'].notna()) &
        (cohort['deathtime'] <= cohort['obs_end_time'])
    )
  early_death_ids = cohort[early_death_mask]['stay_id'].tolist()
  invalid_stay_ids.update(early_death_ids)
  print(f"Phát hiện {len(early_death_ids)} bệnh nhân tử vong TRƯỚC/TRONG 24h đầu ICU.")

  # Loai bỏ các bản ghi vi phạm khỏi Cohort chính thức
  clean_cohort = cohort[~cohort['stay_id'].isin(invalid_stay_ids)].reset_index(drop=True)

  print(f"=== KHỞI TẠO COHORT THÀNH CÔNG ===")
  print(f"Số lượng bệnh nhân hợp lệ cuối cùng: {len(clean_cohort)}")
  print(f"Thời gian nằm ICU trung bình: {cohort['icu_los_days'].mean():.2f} ngày")
  print(f"Tỷ lệ tử vong (Mortality Rate): {clean_cohort['hospital_expire_flag'].mean():.2%}")

  return clean_cohort

In [5]:
# ── CELL 3: xây dựng cohort ───────────────────────────────
print("Loading core tables...")
patients   = load("hosp", "patients.csv.gz")      # Thông tin bệnh nhân
admissions = load("hosp", "admissions.csv.gz")    # Thông tin nhập viện
icustays   = load("icu",  "icustays.csv.gz")      # Thông tin lần nằm

cohort = build_and_val_icu_cohort(patients, admissions, icustays)

Loading core tables...
  ✓ hosp/patients.csv.gz: 100 rows × 6 cols
  ✓ hosp/admissions.csv.gz: 275 rows × 16 cols
  ✓ icu/icustays.csv.gz: 140 rows × 8 cols
Tổng số lượt ICU ban đầu: 140
-> Sau khi lọc người trưởng thành (>=18t): 140 ca
-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: 100 ca
-> Sau khi lọc ICU stay > 1 ngày: 85 ca
=== BẮT ĐẦU THỰC HIỆN Kiểm tra logic ===
Tổng số dòng: 85 | Số stay_id duy nhất: 85
Phát hiện 13 lượt ICU vi phạm thứ tự thời gian sinh lý.
Phát hiện 0 bệnh nhân tử vong TRƯỚC/TRONG 24h đầu ICU.
=== KHỞI TẠO COHORT THÀNH CÔNG ===
Số lượng bệnh nhân hợp lệ cuối cùng: 72
Thời gian nằm ICU trung bình: 4.27 ngày
Tỷ lệ tử vong (Mortality Rate): 1.39%


In [6]:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,hospital_expire_flag,insurance,race,gender,anchor_age,anchor_year,icu_los_days,icu_los_hours,obs_end_time,age_group
0,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,2157-11-18 22:56:00,2157-11-25 18:00:00,...,0,Other,WHITE,F,55,2157,1.118032,26.832778,2157-11-21 19:18:02,51-65
1,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,2110-04-11 15:08:00,2110-04-14 15:00:00,...,0,Other,WHITE,F,46,2110,1.338588,32.126111,2110-04-12 15:52:22,31-50
2,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134,2156-04-12 14:16:00,2156-04-29 16:26:00,...,0,Medicare,WHITE,F,80,2155,4.981134,119.547222,2156-04-13 16:24:18,66-80
3,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,2141-05-22 20:17:00,2141-05-29 17:41:00,...,0,Medicare,UNKNOWN,M,81,2141,5.087512,122.100278,2141-05-23 20:18:01,80+
4,10002930,25696644,37049133,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2196-04-14 13:40:00,2196-04-15 16:54:44,1.135231,2196-04-14 12:25:00,2196-04-17 15:28:00,...,0,Medicare,BLACK/AFRICAN AMERICAN,F,48,2193,1.135231,27.245556,2196-04-15 13:40:00,31-50


In [7]:
cohort.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'admittime', 'dischtime', 'admission_type',
       'admission_location', 'edregtime', 'edouttime', 'deathtime',
       'hospital_expire_flag', 'insurance', 'race', 'gender', 'anchor_age',
       'anchor_year', 'icu_los_days', 'icu_los_hours', 'obs_end_time',
       'age_group'],
      dtype='object')

### **Chia thành các tập train/test/val**

Mục đích nhằm tránh việc Dữ liệu bị rò rỉ ta sẽ chia tập train / test / val sau khi đã tạo bảng cohort
- Sau đó thực hiện join với các bảng tiếp theo để lấy ra các chỉ số lâm sàng trong khung cửa số
- Thực hiện tính toán sinh thêm đặc trưng sẽ thực hiện với tập train với tập test và val thì chỉ đươc dùng để đánh giá thuật toán

In [9]:
def split_dataset_by_subject(
    df: pd.DataFrame,
    subject_col: str = 'subject_id',
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    random_state: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Chia dataset thành 3 tập Train/Val/Test theo `subject_id` để chống Data Leakage.
    Tỷ lệ mặc định: 70% Train - 15% Val - 15% Test.
    """
    assert np.isclose(train_ratio + val_ratio + test_ratio, 1.0), "Tổng các tỷ lệ phải bằng 1.0"

    # 1. Lấy danh sách ID bệnh nhân duy nhất
    unique_subjects = df[subject_col].unique()

    # 2. Xáo trộn danh sách subject_id với seed cố định
    np.random.seed(random_state)
    np.random.shuffle(unique_subjects)

    # 3. Tính toán mốc cắt
    n_total = len(unique_subjects)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)

    # 4. Phân bổ subject_id vào 3 tập độc lập
    train_subjects = set(unique_subjects[:n_train])
    val_subjects = set(unique_subjects[n_train : n_train + n_val])
    test_subjects = set(unique_subjects[n_train + n_val:])

    # 5. Lọc dữ liệu DataFrame theo danh sách subject_id
    train_df = df[df[subject_col].isin(train_subjects)].reset_index(drop=True)
    val_df = df[df[subject_col].isin(val_subjects)].reset_index(drop=True)
    test_df = df[df[subject_col].isin(test_subjects)].reset_index(drop=True)

    # 6. Kiểm tra an toàn
    overlap = train_subjects.intersection(val_subjects) or train_subjects.intersection(test_subjects)
    assert len(overlap) == 0, "LỖI LÊU RÒ DỮ LIỆU: Có bệnh nhân xuất hiện ở nhiều tập!"

    print(f"=== KẾT QUẢ CHIA TẬP DỮ LIỆU ===")
    print(f"Tổng số bệnh nhân (Subjects): {n_total}")
    print(f"- Train: {len(train_subjects)} subjects ({len(train_df)} stays / {len(train_df)/len(df)*100:.1f}%)")
    print(f"- Val  : {len(val_subjects)} subjects ({len(val_df)} stays / {len(val_df)/len(df)*100:.1f}%)")
    print(f"- Test : {len(test_subjects)} subjects ({len(test_df)} stays / {len(test_df)/len(df)*100:.1f}%)")

    return train_df, val_df, test_df

In [10]:
# ── CELL 4: Chia tập train, val, test ───────────────────────────────

train_df, val_df, test_df = split_dataset_by_subject(cohort)

=== KẾT QUẢ CHIA TẬP DỮ LIỆU ===
Tổng số bệnh nhân (Subjects): 72
- Train: 50 subjects (50 stays / 69.4%)
- Val  : 10 subjects (10 stays / 13.9%)
- Test : 12 subjects (12 stays / 16.7%)


### **Xây dựng bảng chart vitals từ bảng `chartevents`**


Để có thể xây dựng bảng này ta dựa vào đặc trưng có trong bảng `chartevents`. Các đặc trưng được lựa chọn:
- Nhịp tim - 220045
- Nhịp thở - 220210
- Nhiệt độ cơ thể độ C - 223762
- Nhiệt độ cơ thể độ F - 223761
- Huyết áp tâm thu / tâm trương / trung bình xâm lấn - 220050 / 220051 / 220052 (Trung bình)
<!-- - Huyết áp tâm thu / tâm trương / trung bình không xâm lấn - 220179 / 220180 / 220181 -->
- SpO2 - 220277
- Glucose máu - 225664
- pH - 223830
- FiO2 - 223835
- Công thức máu -  220228 (Hemoglobin) / 220545 (Hematocrit)/ 220546 (WBC)227457 (Platelet)
- Chức năng thận - 220615 (Creatinine) / 225624 (BUN)
<!-- - Điện giải - 227442 (Potassium) / 220645 (Sodium) / 220602 (Chloride) / 227443 (HCO3) / 220635 (Magnesium) -->
- Khí máu - 223830 (pH Arterial) / 220224 (PaO2)/ 220235 (PaCO2) / 224828 (Base Excess) / 220227 (SaO2)
- Chức năng gan - 220644 (ALT) / 220587 (AST) / 225690 (Bilirubin Toàn phần)227456 (Albumin) / 225612 (Alkaline Phosphate)
- Thang điểm Glagsgow - 223900 (Verbal) / 223901 (Motor) / 220739 (Eye)
<!-- - Marker tim - 227429 (Troponin-T) / 227445 (CK-MB)227446 (BNP) -->
- Nội tiết - 228236 (Insulin) / 227463 (Cortisol)
- Cân nặng - 224639 (Cân nặng hàng ngày) / 226512 (Cân nặng nhập viện Kg)
- Chiều cao - 226707 / 226730

Ta sẽ giới hạn cửa số quan sát là từ 24h trở đi với `intime` là thời gian làm mốc. Tức là sẽ lấy trong khoảng từ `intime` <= `charttime` <= `intiem` + 24h. Với `charttime` là thời gian lấy mẫu
- Do tính chất và đặc điểm của bảng này là Đo liên tục (mỗi 15p/1 lần) nên ta sẽ lấy các mục có khả năng thay đổi, diễn biến nhiều lần

In [39]:
def extract_chartevents_features(
    events_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    item_mapping: Dict[int, str],
    outlier_bounds: Dict[str, Tuple[float, float]],
    return_rag_df: bool = True
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Xử lý chartevents trong 24h ICU:
    1. Lọc mốc thời gian 24h & tính delta_hours.
    2. Quy đổi đơn vị đo (F -> C, lbs -> kg, inches -> cm).
    3. Lọc nhiễu sinh lý (Outliers).
    4. Trích xuất chỉ số chuỗi thời gian: min, max, mean, first, last, delta, slope cho Tabular ML.
    """
    # 1. Tạo danh sách stay_id cơ sở
    result_ids = pd.DataFrame({'stay_id': cohort_df['stay_id'].unique()})

    # Merge với cohort lấy intime để khóa cửa sổ 24h
    df = events_df.merge(
        cohort_df[['stay_id', 'intime']],
        on='stay_id',
        how='inner'
    )

    # Lọc mốc thời gian: intime <= charttime <= intime + 24h
    df['charttime'] = pd.to_datetime(df['charttime'])
    df['intime'] = pd.to_datetime(df['intime'])
    df['obs_end_time'] = df['intime'] + pd.Timedelta(hours=24)

    mask_time = (df['charttime'] >= df['intime']) & (df['charttime'] <= df['obs_end_time'])
    df = df[mask_time].copy()

    # Tính mốc thời gian tương đối delta_hours (tính bằng giờ kể từ intime)
    df['delta_hours'] = (df['charttime'] - df['intime']).dt.total_seconds() / 3600.0

    # Ánh xạ mã itemid sang tên biến lâm sàng
    df = df[df['itemid'].isin(item_mapping.keys())].copy()
    df['feature_name'] = df['itemid'].map(item_mapping)
    df['valuenum'] = pd.to_numeric(df['valuenum'], errors='coerce')
    df = df.dropna(subset=['valuenum']).copy()

    # ---------------------------------------------------------
    # BƯỚC 1: CHUYỂN ĐỔI ĐƠN VỊ ĐO CHUẨN
    # ---------------------------------------------------------
    if 'valueuom' in df.columns:
        uom_str = df['valueuom'].astype(str).str.lower()

        # Fahrenheit -> Celsius
        f_mask = uom_str.str.contains('f', na=False) | (df['feature_name'].str.contains('temp', case=False) & (df['valuenum'] > 50))
        df.loc[f_mask, 'valuenum'] = (df.loc[f_mask, 'valuenum'] - 32.0) * 5.0 / 9.0

        # lbs -> kg
        lbs_mask = uom_str.str.contains('lb', na=False)
        df.loc[lbs_mask, 'valuenum'] = df.loc[lbs_mask, 'valuenum'] * 0.453592

        # inches -> cm
        inch_mask = uom_str.str.contains('inch|in', na=False)
        df.loc[inch_mask, 'valuenum'] = df.loc[inch_mask, 'valuenum'] * 2.54

    # ---------------------------------------------------------
    # BƯỚC 2: LỌC NHIỄU SINH LÝ (OUTLIERS)
    # ---------------------------------------------------------
    for feat_name, (min_val, max_val) in outlier_bounds.items():
        feat_mask = df['feature_name'] == feat_name
        invalid_mask = feat_mask & ((df['valuenum'] < min_val) | (df['valuenum'] > max_val))
        df.loc[invalid_mask, 'valuenum'] = np.nan

    df = df.dropna(subset=['valuenum']).copy()

    # ---------------------------------------------------------
    # BƯỚC 3: TÍNH TOÁN FIRST, LAST, DELTA, SLOPE, MIN, MAX, MEAN
    # ---------------------------------------------------------
    # Sắp xếp theo thời gian đo
    df = df.sort_values(by=['stay_id', 'feature_name', 'charttime'])

    feature_dfs = []
    rag_summaries = {sid: [] for sid in result_ids['stay_id']}

    grouped = df.groupby(['stay_id', 'feature_name'])

    for (sid, feat_name), group in grouped:
        v_min = group['valuenum'].min()
        v_max = group['valuenum'].max()
        v_mean = group['valuenum'].mean()

        v_first = group['valuenum'].iloc[0]
        v_last = group['valuenum'].iloc[-1]

        t_first = group['delta_hours'].iloc[0]
        t_last = group['delta_hours'].iloc[-1]

        v_delta = v_last - v_first
        time_diff = t_last - t_first

        # Slope = Delta / Delta_time (giờ). Nếu chỉ có 1 điểm đo thì Slope = 0.0
        v_slope = (v_delta / time_diff) if time_diff > 0 else 0.0

        # Lưu thông tin cho Tabular
        feature_dfs.append({
            'stay_id': sid,
            'feature_name': feat_name,
            'min': v_min,
            'max': v_max,
            'mean': v_mean,
            'first': v_first,
            'last': v_last,
            'delta': v_delta,
            'slope': v_slope
        })

        # Lưu thông tin diễn tiến cho RAG
        if return_rag_df:
            trend_str = "increasing" if v_delta > 0 else ("decreasing" if v_delta < 0 else "stable")
            rag_summaries[sid].append(
                f"{feat_name.replace('_', ' ').title()}: init {v_first:.1f}, end {v_last:.1f} (range {v_min:.1f}-{v_max:.1f}, trend {trend_str}, slope {v_slope:+.2f}/hr)"
            )

    # ---------------------------------------------------------
    # BƯỚC 4: RESHAPE / UNSTACK SANG MA TRẬN TABULAR (1 DÒNG PER STAY_ID)
    # ---------------------------------------------------------
    if feature_dfs:
        stats_df = pd.DataFrame(feature_dfs)

        # Pivot table để trải rộng các chỉ số thành các cột phẳng
        df_tabular = stats_df.pivot(
            index='stay_id',
            columns='feature_name',
            values=['min', 'max', 'mean', 'first', 'last', 'delta', 'slope']
        )

        # Đổi tên cột dạng: {feature_name}_{stat} (VD: heart_rate_mean, heart_rate_slope)
        df_tabular.columns = [f"{feat}_{stat}" for stat, feat in df_tabular.columns]
        df_tabular = df_tabular.reset_index()

        # Merge lại với danh sách stay_id gốc để đảm bảo đủ số lượng mẫu
        df_tabular = result_ids.merge(df_tabular, on='stay_id', how='left')
    else:
        df_tabular = result_ids.copy()
    return df_tabular

In [12]:
# ── CELL 5: xây dựng chart vitals ───────────────────────────────
# Định nghĩa mapping itemid
item_mapping = {
    # Sinh hiệu cơ bản & Hô hấp
    220045: "heart_rate",        # Nhịp tim (bpm)
    220210: "resp_rate",         # Nhịp thở (lần/phút)
    223762: "temp_c",            # Nhiệt độ độ C (°C)
    223761: "temp_f",            # Nhiệt độ độ F (°F)
    220277: "spo2",              # Nồng độ Oxy máu SpO2 (%)
    223835: "fio2",              # Tỷ lệ Oxy hít vào FiO2 (%)

    # Huyết áp xâm lấn (Arterial Line)
    220050: "sbp_line",          # Huyết áp tâm thu xâm lấn (mmHg)
    220051: "dbp_line",          # Huyết áp tâm trương xâm lấn (mmHg)
    220052: "mbp_line",          # Huyết áp trung bình xâm lấn (mmHg)

    # Huyết áp không xâm lấn (NIBP) - Giữ lại để tránh thiếu dữ liệu
    220179: "sbp_nibp",          # Huyết áp tâm thu NIBP (mmHg)
    220180: "dbp_nibp",          # Huyết áp tâm trương NIBP (mmHg)
    220181: "mbp_nibp",          # Huyết áp trung bình NIBP (mmHg)

    # Đường huyết
    225664: "glucose",           # Glucose máu (mg/dL)

    # Khí máu động mạch (Arterial Blood Gas)
    223830: "ph",                # pH máu động mạch
    220224: "pao2",              # Áp suất một phần Oxy PaO2 (mmHg)
    220235: "paco2",             # Áp suất một phần CO2 PaCO2 (mmHg)
    224828: "base_excess",       # Kiềm dư Base Excess (mEq/L)
    220227: "sao2",              # Độ bão hòa Oxy động mạch SaO2 (%)

    # Công thức máu (CBC)
    220228: "hemoglobin",        # Hemoglobin (g/dL)
    220545: "hematocrit",        # Hematocrit (%)
    220546: "wbc",               # Bạch cầu WBC (K/uL)
    227457: "platelet",          # Tiểu cầu Platelets (K/uL)

    # Chức năng thận
    220615: "creatinine",        # Creatinine (mg/dL)
    225624: "bun",               # Blood Urea Nitrogen (mg/dL)

    # Chức năng gan
    220644: "alt",               # Alanine Aminotransferase ALT (U/L)
    220587: "ast",               # Aspartate Aminotransferase AST (U/L)
    225690: "bilirubin_total",   # Bilirubin toàn phần (mg/dL)
    227456: "albumin",           # Albumin (g/dL)
    225612: "alp",               # Alkaline Phosphatase (U/L)

    # Thang điểm hôn mê Glasgow (GCS)
    223900: "gcs_verbal",        # GCS - Đáp ứng lời nói (1 - 5)
    223901: "gcs_motor",         # GCS - Đáp ứng vận động (1 - 6)
    220739: "gcs_eye",           # GCS - Mở mắt (1 - 4)

    # Nội tiết
    228236: "insulin",           # Insulin (uIU/mL)
    227463: "cortisol",          # Cortisol (ug/dL)

    # Trọng lượng & Chiều cao
    224639: "weight_daily",      # Cân nặng hàng ngày (kg)
    226512: "weight_admit",      # Cân nặng khi nhập viện (kg)
    226707: "height_inch",       # Chiều cao (Inches)
    226730: "height_cm"          # Chiều cao (cm)
}

# Định nghĩa khoảng outlier
outlier_bounds = {
    # Sinh hiệu & Hô hấp
    "heart_rate": (20.0, 250.0),
    "resp_rate": (4.0, 60.0),
    "temp_c": (25.0, 43.0),
    "temp_f": (77.0, 109.4),
    "spo2": (50.0, 100.0),
    "fio2": (21.0, 100.0),        # Trong chartevents, FiO2 thường lưu dạng % (21-100)

    # Huyết áp
    "sbp_line": (40.0, 280.0),
    "dbp_line": (20.0, 150.0),
    "mbp_line": (20.0, 200.0),
    "sbp_nibp": (40.0, 280.0),
    "dbp_nibp": (20.0, 150.0),
    "mbp_nibp": (20.0, 200.0),

    # Đường huyết & Khí máu
    "glucose": (10.0, 2000.0),
    "ph": (6.5, 7.8),
    "pao2": (20.0, 700.0),
    "paco2": (10.0, 150.0),
    "base_excess": (-30.0, 30.0),
    "sao2": (50.0, 100.0),

    # Huyết học (CBC)
    "hemoglobin": (2.0, 25.0),
    "hematocrit": (5.0, 75.0),
    "wbc": (0.1, 200.0),
    "platelet": (5.0, 2000.0),

    # Chức năng thận
    "creatinine": (0.1, 30.0),
    "bun": (1.0, 250.0),

    # Chức năng gan
    "alt": (1.0, 10000.0),
    "ast": (1.0, 10000.0),
    "bilirubin_total": (0.1, 80.0),
    "albumin": (0.5, 7.0),
    "alp": (5.0, 5000.0),

    # GCS
    "gcs_verbal": (1.0, 5.0),
    "gcs_motor": (1.0, 6.0),
    "gcs_eye": (1.0, 4.0),

    # Nội tiết
    "insulin": (0.1, 500.0),
    "cortisol": (0.1, 200.0),

    # Thể trạng
    "weight_daily": (20.0, 300.0),
    "weight_admit": (20.0, 300.0),
    "height_inch": (20.0, 100.0),
    "height_cm": (50.0, 250.0)
}

print("Loading chartevents...")
chartevents   = load("icu",  "chartevents.csv.gz")

Loading chartevents...
  ✓ icu/chartevents.csv.gz: 668,862 rows × 11 cols


In [45]:
vital_chart = extract_chartevents_features(chartevents, train_df, item_mapping, outlier_bounds)

In [46]:
vital_chart.head()

,stay_id,albumin_min,alp_min,alt_min,ast_min,base_excess_min,bilirubin_total_min,bun_min,cortisol_min,creatinine_min,...,platelet_slope,resp_rate_slope,sao2_slope,sbp_line_slope,sbp_nibp_slope,spo2_slope,temp_c_slope,wbc_slope,weight_admit_slope,weight_daily_slope
0,37067082,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,0.4,...,0.000000,0.447906,NaN,NaN,-0.253343,-0.126671,NaN,0.000000,NaN,NaN
1,36753294,NaN,84.0,44.0,210.0,NaN,0.4,30.0,NaN,1.4,...,0.742857,-0.120952,NaN,NaN,-1.692525,-0.426136,NaN,-0.474286,NaN,0.000000
2,37049133,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,0.6,...,0.000000,0.109561,NaN,NaN,0.086331,0.043197,NaN,0.000000,NaN,0.000000
3,35514836,NaN,NaN,NaN,NaN,-3.0,NaN,17.0,NaN,0.9,...,0.000000,-0.450554,NaN,-1.705539,-2.523568,-0.086957,NaN,0.000000,NaN,0.066667
4,32128372,2.7,36.0,15.0,45.0,-19.0,3.0,9.0,NaN,0.7,...,-3.359528,0.110195,-1.674419,0.433839,1.946403,0.000000,NaN,0.129666,NaN,0.170000


In [47]:
vital_chart.shape

(50, 246)

### **Xây dựng bảng lab tests từ bảng `labevents`**


Để có thể xây dựng bảng này ta dựa vào đặc trưng có trong bảng `labevents`. Các đặc trưng được lựa chọn:
- Nhiệt độ cơ thể - 50825
- SpO2 / Bão hòa O2 - 50817
- Glucose máu - 50931 (Huyết tương )/ 50809 (Máu toàn phần)
- pH - 50820
- Công thức máu -  51221 (Hematocrit) / 51222 (Hemoglobin) / 51301 (Bạch cầu - WBC) / 51265 (Tiểu cầu) / 51279 (Hồng cầu - RBC)
- Chức năng thận - 50912 (Creatinine) / 51006 (BUN)
- Điện giải - 50971 (Kali / Potassium) / 50983 (Natri / Sodium) / 50902 (Clo / Chloride) / 50882 (Bicarbonate / HCO3) / 50893 (Calci toàn phần) / 50808 (Calci tự do) / 50960 (Magie) / 50970 (Phosphate)
- Khí máu - 50821 (pO2) / 50818 (pCO2) / 50802 (Base Excess) / 50820 (pH) / 50868 (Anion Gap)
- Chức năng gan - 50861 (ALT / SGPT) / 50878 (AST / SGOT) / 50863 (Alkaline Phosphatase) / 50885 (Bilirubin toàn phần) / 50862 (Albumin) / 50976 (Protein toàn phần)
- Marker tim - 51003 (Troponin T) / 50911 (CK-MB) / 50910 (CK tổng) / 50963 (NT-proBNP)
- Nội tiết - 50993 (TSH) / 50995 (Free T4) / 51001 (T3) / 50909 (Cortisol) / 50965 (PTH)

In [41]:
def extract_labevents_features(
    labevents_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    item_mapping: Dict[int, str],
    outlier_bounds: Dict[str, Tuple[float, float]],
    return_rag_df: bool = True
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Xử lý labevents trong 24h ICU:
    1. Lọc mốc thời gian 24h theo intime & tính delta_hours.
    2. Lọc nhiễu sinh lý (Outliers).
    3. Trích xuất chỉ số chuỗi thời gian: min, max, mean, first, last, delta, slope cho Tabular ML.
    """
    # 1. Khởi tạo danh sách stay_id cơ sở từ cohort
    result_ids = pd.DataFrame({'stay_id': cohort_df['stay_id'].unique()})

    # Tối ưu bộ nhớ: Lọc các itemid quan tâm TRƯỚC khi Merge
    df = labevents_df[labevents_df['itemid'].isin(item_mapping.keys())].copy()

    # Merge với cohort lấy stay_id và intime
    # Lưu ý: labevents kết nối với đợt nằm viện qua subject_id và hadm_id
    df = df.merge(
        cohort_df[['stay_id', 'subject_id', 'hadm_id', 'intime']],
        on=['subject_id', 'hadm_id'],
        how='inner'
    )

    # Lọc mốc thời gian: intime <= charttime <= intime + 24h
    df['charttime'] = pd.to_datetime(df['charttime'])
    df['intime'] = pd.to_datetime(df['intime'])
    df['obs_end_time'] = df['intime'] + pd.Timedelta(hours=24)

    mask_time = (df['charttime'] >= df['intime']) & (df['charttime'] <= df['obs_end_time'])
    df = df[mask_time].copy()

    # Tính mốc thời gian tương đối delta_hours (tính bằng giờ kể từ intime)
    df['delta_hours'] = (df['charttime'] - df['intime']).dt.total_seconds() / 3600.0

    # Ánh xạ mã itemid sang tên biến xét nghiệm
    df['feature_name'] = df['itemid'].map(item_mapping)
    df['valuenum'] = pd.to_numeric(df['valuenum'], errors='coerce')
    df = df.dropna(subset=['valuenum']).copy()

    # ---------------------------------------------------------
    # BƯỚC 1: LỌC NHIỄU SINH LÝ (OUTLIERS)
    # ---------------------------------------------------------
    for feat_name, (min_val, max_val) in outlier_bounds.items():
        feat_mask = df['feature_name'] == feat_name
        invalid_mask = feat_mask & ((df['valuenum'] < min_val) | (df['valuenum'] > max_val))
        df.loc[invalid_mask, 'valuenum'] = np.nan

    df = df.dropna(subset=['valuenum']).copy()

    # ---------------------------------------------------------
    # BƯỚC 2: TÍNH TOÁN FIRST, LAST, DELTA, SLOPE, MIN, MAX, MEAN
    # ---------------------------------------------------------
    # Sắp xếp theo thứ tự thời gian xét nghiệm
    df = df.sort_values(by=['stay_id', 'feature_name', 'charttime'])

    feature_dfs = []
    rag_summaries = {sid: [] for sid in result_ids['stay_id']}

    grouped = df.groupby(['stay_id', 'feature_name'])

    for (sid, feat_name), group in grouped:
        v_min = group['valuenum'].min()
        v_max = group['valuenum'].max()
        v_mean = group['valuenum'].mean()

        v_first = group['valuenum'].iloc[0]
        v_last = group['valuenum'].iloc[-1]

        t_first = group['delta_hours'].iloc[0]
        t_last = group['delta_hours'].iloc[-1]

        v_delta = v_last - v_first
        time_diff = t_last - t_first

        # Slope = Delta / Delta_time (giờ). Nếu chỉ xét nghiệm 1 lần trong 24h -> Slope = 0.0
        v_slope = (v_delta / time_diff) if time_diff > 0 else 0.0

        # Lưu thuộc tính cho Tabular Dataframe
        feature_dfs.append({
            'stay_id': sid,
            'feature_name': feat_name,
            'min': v_min,
            'max': v_max,
            'mean': v_mean,
            'first': v_first,
            'last': v_last,
            'delta': v_delta,
            'slope': v_slope
        })

        # Lưu tóm tắt diễn tiến cho RAG context
        if return_rag_df:
            trend_str = "increasing" if v_delta > 0 else ("decreasing" if v_delta < 0 else "stable")
            # Kiểm tra nếu chỉ có 1 lần làm xét nghiệm
            if len(group) == 1:
                summary_text = f"{feat_name.replace('_', ' ').title()}: single test at {t_first:.1f}h = {v_first:.2f}"
            else:
                summary_text = (
                    f"{feat_name.replace('_', ' ').title()}: init {v_first:.2f}, end {v_last:.2f} "
                    f"(min/max {v_min:.2f}-{v_max:.2f}, trend {trend_str}, slope {v_slope:+.3f}/hr)"
                )
            rag_summaries[sid].append(summary_text)

    # ---------------------------------------------------------
    # BƯỚC 3: UNSTACK SANG MA TRẬN TABULAR (1 DÒNG PER STAY_ID)
    # ---------------------------------------------------------
    if feature_dfs:
        stats_df = pd.DataFrame(feature_dfs)

        # Unstack các chỉ số thống kê thành cột ngang
        df_tabular = stats_df.pivot(
            index='stay_id',
            columns='feature_name',
            values=['min', 'max', 'mean', 'first', 'last', 'delta', 'slope']
        )

        # Đổi tên cột dạng: {lab_feature_name}_{stat} (VD: lactate_last, creatinine_slope)
        df_tabular.columns = [f"{feat}_{stat}" for stat, feat in df_tabular.columns]
        df_tabular = df_tabular.reset_index()

        # Merge với danh sách stay_id chuẩn của cohort
        df_tabular = result_ids.merge(df_tabular, on='stay_id', how='left')
    else:
        df_tabular = result_ids.copy()
    return df_tabular

In [19]:
# ── CELL 6: xây dựng lab tests ───────────────────────────────
# Định nghĩa mapping itemid
lab_item_mapping = {
    # Sinh hiệu & Khí máu cơ bản
    50825: "temp_c_lab",          # Nhiệt độ cơ thể (°C)
    50817: "spo2_lab",            # Độ bão hòa Oxy (O2 Saturation / SpO2 %)
    50820: "ph",                  # pH máu

    # Đường huyết
    50931: "glucose_plasma",      # Glucose Huyết tương (mg/dL)
    50809: "glucose_blood",       # Glucose Máu toàn phần (mg/dL)

    # Huyết học (Công thức máu - CBC)
    51221: "hematocrit",          # Hematocrit (%)
    51222: "hemoglobin",          # Hemoglobin (g/dL)
    51301: "wbc",                 # Bạch cầu WBC (K/uL)
    51265: "platelet",            # Tiểu cầu Platelets (K/uL)
    51279: "rbc",                 # Hồng cầu RBC (m/uL)

    # Chức năng thận
    50912: "creatinine",          # Creatinine (mg/dL)
    51006: "bun",                 # Blood Urea Nitrogen (mg/dL)

    # Điện giải đồ
    50971: "potassium",           # Kali / Potassium (mEq/L)
    50983: "sodium",              # Natri / Sodium (mEq/L)
    50902: "chloride",            # Clo / Chloride (mEq/L)
    50882: "bicarbonate",         # Bicarbonate / HCO3 (mEq/L)
    50893: "calcium_total",       # Calci toàn phần (mg/dL)
    50808: "calcium_free",        # Calci ion hóa / tự do (mmol/L)
    50960: "magnesium",           # Magie (mg/dL)
    50970: "phosphate",           # Phosphate (mg/dL)

    # Khí máu động mạch / tĩnh mạch (ABG/VBG)
    50821: "po2",                 # Áp suất một phần Oxy pO2 (mmHg)
    50818: "pco2",                # Áp suất một phần CO2 pCO2 (mmHg)
    50802: "base_excess",         # Kiềm dư Base Excess (mEq/L)
    50868: "anion_gap",           # Khoảng trống Anion Gap (mEq/L)

    # Chức năng gan
    50861: "alt",                 # Alanine Aminotransferase ALT (U/L)
    50878: "ast",                 # Aspartate Aminotransferase AST (U/L)
    50863: "alp",                 # Alkaline Phosphatase (U/L)
    50885: "bilirubin_total",     # Bilirubin toàn phần (mg/dL)
    50862: "albumin",             # Albumin (g/dL)
    50976: "protein_total",       # Protein toàn phần (g/dL)

    # Men tim & Marker sinh học (Cardiac Markers)
    51003: "troponin_t",          # Troponin T (ng/mL)
    50911: "ck_mb",               # CK-MB (ng/mL)
    50910: "ck_total",            # CK tổng (U/L)
    50963: "nt_probnp",           # NT-proBNP (pg/mL)

    # Nội tiết (Endocrinology)
    50993: "tsh",                 # Thyroid-Stimulating Hormone (uIU/mL)
    50995: "free_t4",             # Free T4 (ng/dL)
    51001: "t3",                  # Triiodothyronine T3 (ng/dL)
    50909: "cortisol",            # Cortisol (ug/dL)
    50965: "pth"                  # Parathyroid Hormone PTH (pg/mL)
}

# Định nghĩa khoảng outlier
lab_outlier_bounds = {
    # Sinh hiệu & Khí máu
    "temp_c_lab": (25.0, 43.0),
    "spo2_lab": (50.0, 100.0),
    "ph": (6.5, 7.8),

    # Đường huyết
    "glucose_plasma": (10.0, 2000.0),
    "glucose_blood": (10.0, 2000.0),

    # Huyết học (CBC)
    "hematocrit": (5.0, 75.0),
    "hemoglobin": (2.0, 25.0),
    "wbc": (0.1, 200.0),
    "platelet": (5.0, 2000.0),
    "rbc": (0.5, 10.0),

    # Chức năng thận
    "creatinine": (0.1, 30.0),
    "bun": (1.0, 250.0),

    # Điện giải đồ
    "potassium": (1.0, 10.0),
    "sodium": (90.0, 180.0),
    "chloride": (60.0, 160.0),
    "bicarbonate": (2.0, 60.0),
    "calcium_total": (2.0, 20.0),
    "calcium_free": (0.2, 3.0),
    "magnesium": (0.2, 10.0),
    "phosphate": (0.2, 20.0),

    # Khí máu
    "po2": (20.0, 700.0),
    "pco2": (10.0, 150.0),
    "base_excess": (-30.0, 30.0),
    "anion_gap": (0.0, 60.0),

    # Chức năng gan
    "alt": (1.0, 10000.0),
    "ast": (1.0, 10000.0),
    "alp": (5.0, 5000.0),
    "bilirubin_total": (0.1, 80.0),
    "albumin": (0.5, 7.0),
    "protein_total": (1.0, 15.0),

    # Men tim
    "troponin_t": (0.01, 100.0),
    "ck_mb": (0.1, 1000.0),
    "ck_total": (5.0, 50000.0),
    "nt_probnp": (10.0, 100000.0),

    # Nội tiết
    "tsh": (0.01, 100.0),
    "free_t4": (0.1, 10.0),
    "t3": (10.0, 800.0),
    "cortisol": (0.1, 200.0),
    "pth": (1.0, 3000.0)
}

labevents = load("hosp", "labevents.csv.gz")

  ✓ hosp/labevents.csv.gz: 107,727 rows × 16 cols


In [42]:
lab_test = extract_labevents_features(labevents, train_df, lab_item_mapping, lab_outlier_bounds)

In [43]:
lab_test.head()

,stay_id,albumin_min,alp_min,alt_min,anion_gap_min,ast_min,base_excess_min,bicarbonate_min,bilirubin_total_min,bun_min,...,po2_slope,potassium_slope,protein_total_slope,rbc_slope,sodium_slope,spo2_lab_slope,temp_c_lab_slope,troponin_t_slope,tsh_slope,wbc_slope
0,37067082,NaN,NaN,NaN,15.0,NaN,NaN,23.0,NaN,9.0,...,NaN,0.000000,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000
1,36753294,NaN,84.0,44.0,19.0,210.0,NaN,14.0,0.4,30.0,...,NaN,-0.014833,NaN,-0.015429,0.148331,NaN,NaN,0.156388,NaN,-0.474286
2,37049133,NaN,NaN,NaN,11.0,NaN,NaN,21.0,NaN,15.0,...,NaN,0.000000,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000
3,35514836,NaN,NaN,NaN,14.0,NaN,-3.0,26.0,NaN,17.0,...,-0.978261,0.033708,NaN,0.000000,0.112360,NaN,0.000000,NaN,NaN,0.000000
4,32128372,2.7,36.0,15.0,24.0,45.0,-19.0,10.0,3.0,9.0,...,-3.423555,-0.031250,NaN,0.053635,-0.078125,-6.851211,0.078478,NaN,NaN,0.129666


In [44]:
lab_test.shape

(50, 253)

### **Xây dựng các đăc trưng liên quan đến Bối cảnh y tế và Tai lượng Bệnh nền của bệnh nhân thông qua 5 bảng còn lại**

**Mục tiêu** Tạo ra đặc trưng mới thêm phần đặc sắt cho mô hình dự đoán cũng như là dữ liệu để LLM có thể tổng hợp và sinh tóm tắt

**Thực hiện thao tác lấy các thuốc để Đánh giá phác đồ điều trị và nguy cơ bị đọc do thuốc**

Bảng được chọn `prescriptions` Dùng thuốc toàn viện / ICU
- Mục tiêu là chỉ lấy các thuốc có khoảng thời gian dùng giao với cửa sổ 24h trong ICU

In [23]:
def extract_raw_prescription_features(
    prescriptions_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    med_keywords: Dict[str, List[str]]
) -> pd.DataFrame:
    """
    Trích xuất cờ nhị phân thô (0/1) ghi nhận đơn thuốc giao thoa với 24h ICU.
    """
    result = pd.DataFrame({'stay_id': cohort_df['stay_id'].unique()})

    rx = prescriptions_df.merge(cohort_df[['stay_id', 'hadm_id', 'intime']], on='hadm_id', how='inner')
    rx['starttime'] = pd.to_datetime(rx['starttime'])
    rx['stoptime'] = pd.to_datetime(rx['stoptime'])

    obs_start = rx['intime']
    obs_end = rx['intime'] + pd.Timedelta(hours=24)

    mask_intersect = (rx['starttime'] <= obs_end) & (rx['stoptime'] >= obs_start)
    rx_valid = rx[mask_intersect].copy()
    rx_valid['drug_lower'] = rx_valid['drug'].astype(str).str.lower()

    for med_group, keywords in med_keywords.items():
        pattern = '|'.join(keywords)
        matched_stays = rx_valid[rx_valid['drug_lower'].str.contains(pattern, na=False)]['stay_id'].unique()
        result[f"raw_has_{med_group}"] = result['stay_id'].isin(matched_stays).astype(int)

    return result

In [25]:
med_keywords = {
    # 1. Thuốc vận mạch & Tăng co bóp cơ tim (Chỉ dấu nguy cơ Shock/Tử vong cao nhất)
    "vasopressors_inotropes": [
        "norepinephrine", "levophed",
        "epinephrine", "adrenaline",
        "vasopressin",
        "dopamine",
        "dobutamine",
        "phenylephrine", "neo-synephrine",
        "milrinone"
    ],

    # 2. Thuốc giãn cơ (Neuromuscular Blockers - Thường dùng khi ARDS rất nặng/Cần thở máy xâm lấn)
    "paralytics": [
        "cisatracurium", "nimbex",
        "vecuronium",
        "rocuronium",
        "pancuronium"
    ],

    # 3. Thuốc an thần & Giảm đau truyền liên tục (Chỉ dấu thở máy/Mê sâu)
    "sedatives_analgesics": [
        "propofol", "diprivan",
        "midazolam", "versed",
        "fentanyl",
        "dexmedetomidine", "precedex",
        "ketamine",
        "lorazepam", "ativan"
    ],

    # 4. Kháng sinh phổ rộng & Antifungal (Chỉ dấu Nhiễm trùng huyết / Septic Shock)
    "broad_spectrum_antibiotics": [
        "vancomycin",
        "piperacillin", "zosyn",
        "meropenem", "imipenem",
        "cefepime",
        "colistin", "polymyxin",
        "fluconazole", "micafungin", "caspofungin"
    ],

    # 5. Thuốc chống đông liều điều trị / Chống sợi huyết (DVT/PE/MI/DIC)
    "anticoagulants_severe": [
        "heparin",  # Lưu ý: Heparin truyền tĩnh mạch liều cao
        "bivalirudin",
        "argatroban",
        "alteplase", "tpa"
    ],

    # 6. Thuốc chống loạn nhịp cấp cứu
    "antiarrhythmics": [
        "amiodarone",
        "diltiazem",
        "adenosine",
        "lidocaine",
        "atropine"
    ],

    # 7. Corticosteroid liều cao/Shock nhiễm khuẩn
    "corticosteroids": [
        "hydrocortisone",
        "methylprednisolone", "solu-medrol",
        "dexamethasone"
    ],

    # 8. Thuốc lợi tiểu & Dịch truyền tĩnh mạch đặc biệt (Suy thận cấp / Quá tải dịch / Phù não)
    "diuretics_fluids": [
        "furosemide", "lasix",
        "bumetanide",
        "albumin",
        "sodium chloride 3%"  # Hypertonic saline
    ]
}

prescriptions   = load("hosp",  "prescriptions.csv.gz")

  ✓ hosp/prescriptions.csv.gz: 18,087 rows × 21 cols


In [26]:
prescriptions_drug = extract_raw_prescription_features(prescriptions, train_df, med_keywords)

In [27]:
prescriptions_drug.head()

,stay_id,raw_has_vasopressors_inotropes,raw_has_paralytics,raw_has_sedatives_analgesics,raw_has_broad_spectrum_antibiotics,raw_has_anticoagulants_severe,raw_has_antiarrhythmics,raw_has_corticosteroids,raw_has_diuretics_fluids
0,37067082,0,0,0,1,1,1,0,0
1,36753294,1,0,0,1,1,1,0,1
2,37049133,0,0,0,0,1,0,0,0
3,35514836,0,0,0,1,1,1,0,0
4,32128372,1,0,1,1,1,0,0,1


**Thực hiện lấy các tiểu sử bệnh nền**

Bảng đươc chọn là `diagnoses_icd` mục tiêu nhằm sinh ra các đặc trưng dạng nhị phân đánh dấu lọc ra các bệnh nền theo chuẩn khi thêm vào llm


In [29]:
def process_prior_comorbidities(
    diagnoses_icd_df: pd.DataFrame,
    admissions_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    icd_patterns: Dict[str, List[str]]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Trích xuất tiền sử bệnh nền TỪ CÁC ĐỢT NHẬP VIỆN TRƯỚC ĐÓ (Chủ động tránh Data Leakage).

    Returns:
        df_ml: Bảng biến nhị phân + tổng số bệnh nền cho ML.
    """
    result = pd.DataFrame({'stay_id': cohort_df['stay_id'].unique()})

    # 1. Tối ưu join: Xác định các đợt nhập viện cũ (hadm_id) kết thúc TRƯỚC thời điểm vào ICU (intime)
    adm_times = admissions_df[['subject_id', 'hadm_id', 'dischtime']].copy()
    adm_times['dischtime'] = pd.to_datetime(adm_times['dischtime'])

    cohort_times = cohort_df[['stay_id', 'subject_id', 'intime']].copy()
    cohort_times['intime'] = pd.to_datetime(cohort_times['intime'])

    # Merge theo subject_id để so sánh thời gian
    merged_adm = adm_times.merge(cohort_times, on='subject_id', how='inner')

    # LỌC ĐIỀU KIỆN CHỐNG LEAKAGE: dischtime của đợt cũ < intime đợt ICU này
    prior_admissions = merged_adm[merged_adm['dischtime'] < merged_adm['intime']][['stay_id', 'hadm_id']].drop_duplicates()

    # 2. Join với diagnoses_icd chỉ trên các hadm_id hợp lệ đã lọc
    prior_diags = diagnoses_icd_df.merge(prior_admissions, on='hadm_id', how='inner')
    prior_diags['icd_code_clean'] = prior_diags['icd_code'].astype(str).str.strip().str.upper()

    # Dictionary lưu bệnh nền theo stay_id phục vụ RAG
    rag_comorbidities = {sid: [] for sid in result['stay_id']}

    # 3. Trích xuất cờ bệnh nền theo Regex tiền tố
    for comorb_name, codes in icd_patterns.items():
        # Chuẩn hóa mã ICD dạng pattern regex tiền tố
        clean_codes = [c.strip().upper() for c in codes]
        pattern = r'^(' + '|'.join(clean_codes) + r')'

        # Bắt các stay_id khớp mã ICD
        matched_mask = prior_diags['icd_code_clean'].str.contains(pattern, na=False, regex=True)
        matched_stays = prior_diags[matched_mask]['stay_id'].unique()

        # Gán giá trị 0/1 cho ML
        col_name = f"prior_{comorb_name}"
        result[col_name] = result['stay_id'].isin(matched_stays).astype(int)

        # Lưu tên bệnh nền cho RAG
        formatted_name = comorb_name.replace('comorbidity_', '').replace('_', ' ').title()
        for sid in matched_stays:
            rag_comorbidities[sid].append(formatted_name)

    # 4. Tạo Feature tổng số lượng bệnh nền (Rất hữu ích cho Tabular ML)
    comorb_cols = [c for c in result.columns if c.startswith('prior_comorbidity_')]
    result['prior_comorbidity_count'] = result[comorb_cols].sum(axis=1)

    return result

In [32]:
icd_patterns = {
    # 1. Suy tim mãn tính (CHF)
    'comorbidity_chf': [
        '428',          # ICD-9: Heart failure
        'I50', 'I110', 'I130', 'I132' # ICD-10
    ],

    # 2. Bệnh thận mãn / Suy thận giai đoạn cuối (CKD / ESRD)
    'comorbidity_ckd': [
        '585', '586',   # ICD-9: Chronic kidney disease
        'N18', 'N19'    # ICD-10
    ],

    # 3. Bệnh phổi tắc nghẽn mãn tính / Hen suyễn (COPD / Asthma)
    'comorbidity_copd': [
        '491', '492', '496', '493', # ICD-9: Chronic bronchitis, emphysema, COPD, asthma
        'J44', 'J41', 'J42', 'J43', 'J45' # ICD-10
    ],

    # 4. Đái tháo đường (Diabetes Mellitus - Bao gồm có và không biến chứng)
    'comorbidity_diabetes': [
        '250',          # ICD-9: Diabetes mellitus
        'E10', 'E11', 'E13' # ICD-10
    ],

    # 5. Xơ gan / Bệnh gan nặng (Cirrhosis / Severe Liver Disease)
    'comorbidity_liver_disease': [
        '571', '572',   # ICD-9: Cirrhosis, chronic liver disease
        'K70', 'K74', 'K72' # ICD-10
    ],

    # 6. Ung thư / Di căn (Malignancy / Metastatic Cancer)
    'comorbidity_cancer': [
        '140', '150', '160', '170', '180', '190', '196', '197', '198', '199', # ICD-9
        'C00', 'C15', 'C30', 'C40', 'C50', 'C60', 'C77', 'C78', 'C79', 'C80'  # ICD-10
    ],

    # 7. Tai biến mạch máu não / Tiền sử Stroke (Cerebrovascular Disease)
    'comorbidity_stroke': [
        '430', '431', '432', '433', '434', '436', '438', # ICD-9
        'I60', 'I61', 'I62', 'I63', 'I69'               # ICD-10
    ],

    # 8. Mạch máu ngoại biên (Peripheral Vascular Disease)
    'comorbidity_pvd': [
        '440', '441', '443', # ICD-9
        'I70', 'I71', 'I73'  # ICD-10
    ],

    # 9. Tăng huyết áp (Hypertension)
    'comorbidity_hypertension': [
        '401', '402', '403', '404', '405', # ICD-9
        'I10', 'I11', 'I12', 'I13', 'I15'  # ICD-10
    ],

    # 10. Suy giảm miễn dịch / HIV (Immunosuppression)
    'comorbidity_immunodeficiency': [
        '042', 'V42',   # ICD-9: HIV, organ transplant
        'B20', 'Z94'    # ICD-10
    ]
}

diagnoses_icd = load("hosp", "diagnoses_icd.csv.gz")
admissions     = load("hosp", "admissions.csv.gz")

  ✓ hosp/diagnoses_icd.csv.gz: 4,506 rows × 5 cols
  ✓ hosp/admissions.csv.gz: 275 rows × 16 cols


In [48]:
icd_comorbidities = process_prior_comorbidities(diagnoses_icd, admissions, train_df, icd_patterns)

In [49]:
icd_comorbidities.head()

,stay_id,prior_comorbidity_chf,prior_comorbidity_ckd,prior_comorbidity_copd,prior_comorbidity_diabetes,prior_comorbidity_liver_disease,prior_comorbidity_cancer,prior_comorbidity_stroke,prior_comorbidity_pvd,prior_comorbidity_hypertension,prior_comorbidity_immunodeficiency,prior_comorbidity_count
0,37067082,0,0,0,0,0,0,0,0,0,0,0
1,36753294,0,0,0,0,0,0,0,0,0,0,0
2,37049133,0,0,0,0,0,0,0,0,0,0,0
3,35514836,0,0,0,0,0,0,0,0,0,0,0
4,32128372,1,1,0,0,0,0,0,0,1,0,3


In [50]:
icd_comorbidities.shape

(50, 12)

**Thực hiện thao tác lấy các đánh giá mức độ can thiệp xâm lấn và hỗ trợ cơ quan**

- Bảng được chọn là `procedureevents` chứa thủ thuật ICU
  - Mục tiêu: Tạo các đặc trưng mới dưới nhãn là nhị phân nếu bệnh nhân có thở máy thì đánh dấu giá trị là 1 ngược lại thì 0


In [34]:
def extract_procedure_features_and_rag_text(
    procedureevents_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    procedure_mapping: Dict[str, List[int]]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Trích xuất từ procedureevents trong 24h đầu ICU:
    1. df_ml: Cờ nhị phân (0/1) + Tổng số thủ thuật cho Tabular ML.
    """
    # 1. Khởi tạo DataFrame kết quả
    result = pd.DataFrame({'stay_id': cohort_df['stay_id'].unique()})

    # Merge dữ liệu với cohort
    procs = procedureevents_df.merge(
        cohort_df[['stay_id', 'intime']],
        on='stay_id',
        how='inner'
    )

    # 2. Chuẩn hóa Datetime & Item ID
    procs['intime'] = pd.to_datetime(procs['intime'])
    procs['starttime'] = pd.to_datetime(procs['starttime'])

    # Mốc thời gian 24h đầu ICU
    obs_start = procs['intime']
    obs_end = procs['intime'] + pd.Timedelta(hours=24)

    # Xử lý endtime: Nếu null (thủ thuật đang tiếp diễn), giả định kéo dài qua obs_end
    if 'endtime' in procs.columns:
        procs['endtime'] = pd.to_datetime(procs['endtime']).fillna(obs_end + pd.Timedelta(days=1))
    else:
        procs['endtime'] = procs['starttime']

    # 3. Điều kiện GIAO THOA THỜI GIAN (Overlap) trong 24h đầu ICU
    mask_intersect = (procs['starttime'] <= obs_end) & (procs['endtime'] >= obs_start)
    procs_24h = procs[mask_intersect].copy()

    # Chuẩn hóa itemid về int
    procs_24h['itemid'] = pd.to_numeric(procs_24h['itemid'], errors='coerce').fillna(-1).astype(int)

    # Dictionary lưu thủ thuật theo stay_id cho RAG
    rag_procedures = {sid: [] for sid in result['stay_id']}

    # 4. Trích xuất cờ nhị phân theo mapping itemid
    for proc_name, itemids in procedure_mapping.items():
        # Lọc các stay_id có thực hiện thủ thuật này
        matched_mask = procs_24h['itemid'].isin(itemids)
        matched_stays = procs_24h[matched_mask]['stay_id'].unique()

        # Gán biến nhị phân 0/1 cho ML
        col_name = f"has_{proc_name}"
        result[col_name] = result['stay_id'].isin(matched_stays).astype(int)

        # Lưu tên thủ thuật cho RAG
        formatted_name = proc_name.replace('proc_', '').replace('_', ' ').title()
        for sid in matched_stays:
            rag_procedures[sid].append(formatted_name)

    # 5. Feature tính tổng số lượng thủ thuật nguy cơ cao/xâm lấn được thực hiện
    proc_cols = [c for c in result.columns if c.startswith('has_')]
    result['procedure_count_24h'] = result[proc_cols].sum(axis=1)

    return result

In [35]:
procedure_mapping = {
    # 1. Thở máy xâm lấn / Đặt nội khí quản (Invasive Mechanical Ventilation)
    "proc_invasive_ventilation": [
        225792,  # Invasive Ventilation
        224385,  # Intubation
        225468,  # Extubation (Nếu có rút ống cũng ghi nhận ca nặng)
        225477   # Tracheostomy
    ],

    # 2. Lọc máu liên tục / Thận nhân tạo cấp cứu (CRRT / Hemodialysis)
    "proc_dialysis_crrt": [
        225802,  # Dialysis - CRRT
        225803,  # Dialysis - Hemodialysis
        225805,  # Peritoneal Dialysis
        225441   # Hemofiltration
    ],

    # 3. Đặt đường truyền trung tâm / Động mạch (Arterial Line / Central Line Access)
    "proc_vascular_access": [
        224267,  # Central line insert
        224268,  # PICC line insert
        224270,  # Dialysis Catheter placement
        225204   # Arterial line insert
    ],

    # 4. Bóng đối xung động mạch chủ / Hỗ trợ tuần hoàn (IABP / ECMO / Impella)
    "proc_circulatory_support": [
        225323,  # Intra-Aortic Balloon Pump (IABP)
        225828,  # ECMO
        228220   # Impella
    ],

    # 5. Dẫn lưu màng phổi / Màng tim (Chest Tube / Thoracentesis / Paracentesis)
    "proc_drainage": [
        225183,  # Chest tube insertion
        225432,  # Thoracentesis
        225459,  # Paracentesis
        225181   # Pericardiocentesis
    ],

    # 6. Mở khí quản / Phẫu thuật cấp cứu ICU
    "proc_bronchoscopy_endoscopy": [
        225430,  # Bronchoscopy
        225454   # EGD / GI Endoscopy (Cấp cứu xuất huyết tiêu hóa)
    ],

    # 7. Hồi sức tim phổi / Sốc điện (CPR / Cardioversion)
    "proc_cpr_cardioversion": [
        225463,  # CPR / Resuscitation
        225464   # Cardioversion / Defibrillation
    ]
}

procedureevents  = load("icu",  "procedureevents.csv.gz")

  ✓ icu/procedureevents.csv.gz: 1,468 rows × 22 cols


In [51]:
procedure_features = extract_procedure_features_and_rag_text(procedureevents, train_df, procedure_mapping)

In [52]:
procedure_features.head()

,stay_id,has_proc_invasive_ventilation,has_proc_dialysis_crrt,has_proc_vascular_access,has_proc_circulatory_support,has_proc_drainage,has_proc_bronchoscopy_endoscopy,has_proc_cpr_cardioversion,procedure_count_24h
0,37067082,0,0,0,0,0,0,0,0
1,36753294,0,0,0,0,1,0,0,1
2,37049133,0,0,0,0,0,0,0,0
3,35514836,0,0,0,0,1,0,0,1
4,32128372,1,0,1,0,0,1,0,3


In [53]:
procedure_features.shape

(50, 9)

**Thực hiện thao tác lấy dịch truyền + thuốc vận mạch cũng như nước tiểu + dịch dẫn lưu thông qua 2 bảng**
- `inputevents:` Đánh giá mức độ sốc sức dịch và sự phụ thuộc vào thuốc hỗ trợ tuần hoàn
- `outputevents:` Đánh giá chức năng thận và cân bằng dịch
  - Tính toán tổng thể dục vào dịch ra, nước tiểu, tốc độ nước tiểu (ml/kg/h)
  - Tính cân bằng dịch trong 24h
  - Thực hiện ở các bước tiếp theo có thể là sinh thêm đặc trưng làm cờ nhị phân

In [71]:
import pandas as pd
import numpy as np
from typing import List, Tuple, Optional

def process_inputs_outputs_and_rag_text(
    inputevents_df: pd.DataFrame,
    outputevents_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    urine_itemids: List[int],
    chartevents_df: Optional[pd.DataFrame] = None,
    weight_itemids: List[int] = [226512, 224639, 226531],
    return_rag_df: bool = True,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Tính toán chỉ số Dịch vào/Dịch ra/Cân bằng dịch trong 24h đầu ICU khi cohort KHÔNG CÓ weight_admit.

    1. Tự động lấy cân nặng từ chartevents (nếu có), nếu thiếu -> gán mặc định 70.0 kg.
    2. Lọc dịch vào (inputevents) & dịch ra (outputevents) trong 24h đầu ICU.
    3. Tính Net Balance, tốc độ nước tiểu (mL/kg/h) và các cờ nguy cơ KDIGO (Oliguria, Anuria, Fluid Overload).
    """
    # 1. Khởi tạo DataFrame kết quả dựa trên cohort_df
    result = cohort_df[['stay_id', 'intime']].copy()
    result['intime'] = pd.to_datetime(result['intime'])

    # ---------------------------------------------------------
    # 2. XỬ LÝ CÂN NẶNG (WEIGHT EXTRACTION FROM CHARTEVENTS)
    # ---------------------------------------------------------
    weight_df = pd.DataFrame({'stay_id': result['stay_id'], 'weight_kg': np.nan})

    if chartevents_df is not None and not chartevents_df.empty:
        # Lọc các bản ghi cân nặng trong chartevents
        weights = chartevents_df[chartevents_df['itemid'].isin(weight_itemids)].copy()
        if not weights.empty:
            weights = weights.merge(result[['stay_id', 'intime']], on='stay_id', how='inner')
            weights['charttime'] = pd.to_datetime(weights['charttime'])

            # Lọc trong 24h đầu
            mask_w = (weights['charttime'] >= weights['intime']) & (weights['charttime'] <= weights['intime'] + pd.Timedelta(hours=24))
            weights_24h = weights[mask_w].copy()
            weights_24h['valuenum'] = pd.to_numeric(weights_24h['valuenum'], errors='coerce')

            # Chuẩn hóa lbs -> kg nếu cần
            if 'valueuom' in weights_24h.columns:
                lbs_mask = weights_24h['valueuom'].astype(str).str.lower().str.contains('lb', na=False)
                weights_24h.loc[lbs_mask, 'valuenum'] = weights_24h.loc[lbs_mask, 'valuenum'] * 0.453592

            # Lấy giá trị cân nặng đầu tiên đo được
            first_weight = weights_24h.dropna(subset=['valuenum']).groupby('stay_id')['valuenum'].first().reset_index()
            first_weight.rename(columns={'valuenum': 'weight_kg'}, inplace=True)
            weight_df = weight_df.drop(columns=['weight_kg']).merge(first_weight, on='stay_id', how='left')

    # Merge cân nặng vào result & Impute giá trị vô lý/khuyết bằng 70.0 kg
    result = result.merge(weight_df, on='stay_id', how='left')
    result['weight_kg'] = pd.to_numeric(result['weight_kg'], errors='coerce')
    invalid_weight_mask = (result['weight_kg'].isna()) | (result['weight_kg'] <= 0) | (result['weight_kg'] > 300)
    result.loc[invalid_weight_mask, 'weight_kg'] = 70.0

    # ---------------------------------------------------------
    # 3. XỬ LÝ INPUTEVENTS (Dịch vào 24h)
    # ---------------------------------------------------------
    inputs = inputevents_df.merge(result[['stay_id', 'intime']], on='stay_id', how='inner')
    inputs['starttime'] = pd.to_datetime(inputs['starttime'])

    # Lọc cửa sổ 24h đầu ICU
    mask_in_24h = (inputs['starttime'] >= inputs['intime']) & (inputs['starttime'] <= inputs['intime'] + pd.Timedelta(hours=24))
    inputs_24h = inputs[mask_in_24h].copy()

    # Bọc an toàn: Chỉ cộng các dòng có đơn vị thể tích (mL/ml)
    if 'amountuom' in inputs_24h.columns:
        valid_vol_mask = inputs_24h['amountuom'].astype(str).str.lower().isin(['ml', 'milliliters'])
        inputs_24h = inputs_24h[valid_vol_mask]

    total_input = inputs_24h.groupby('stay_id')['amount'].sum().reset_index()
    total_input.rename(columns={'amount': 'input_total_volume_24h'}, inplace=True)

    # ---------------------------------------------------------
    # 4. XỬ LÝ OUTPUTEVENTS (Dịch ra & Nước tiểu 24h)
    # ---------------------------------------------------------
    outputs = outputevents_df.merge(result[['stay_id', 'intime']], on='stay_id', how='inner')
    outputs['charttime'] = pd.to_datetime(outputs['charttime'])

    # Lọc cửa sổ 24h đầu ICU
    mask_out_24h = (outputs['charttime'] >= outputs['intime']) & (outputs['charttime'] <= outputs['intime'] + pd.Timedelta(hours=24))
    outputs_24h = outputs[mask_out_24h].copy()

    # Chuẩn hóa itemid
    outputs_24h['itemid'] = pd.to_numeric(outputs_24h['itemid'], errors='coerce').fillna(-1).astype(int)
    outputs_24h['value'] = pd.to_numeric(outputs_24h['value'], errors='coerce').fillna(0.0)

    # Tổng Dịch Ra
    total_output = outputs_24h.groupby('stay_id')['value'].sum().reset_index()
    total_output.rename(columns={'value': 'output_total_volume_24h'}, inplace=True)

    # Nước tiểu (Urine Output)
    urine_outputs = outputs_24h[outputs_24h['itemid'].isin(urine_itemids)]
    total_urine = urine_outputs.groupby('stay_id')['value'].sum().reset_index()
    total_urine.rename(columns={'value': 'output_urine_24h'}, inplace=True)

    # ---------------------------------------------------------
    # 5. TÍNH CHỈ SỐ LÂM SÀNG (ML FEATURES)
    # ---------------------------------------------------------
    result = result.merge(total_input, on='stay_id', how='left').fillna({'input_total_volume_24h': 0.0})
    result = result.merge(total_output, on='stay_id', how='left').fillna({'output_total_volume_24h': 0.0})
    result = result.merge(total_urine, on='stay_id', how='left').fillna({'output_urine_24h': 0.0})

    # Cân bằng dịch ròng (Net Fluid Balance)
    result['net_fluid_balance_24h'] = result['input_total_volume_24h'] - result['output_total_volume_24h']

    # Tốc độ nước tiểu (mL/kg/h) dựa trên cân nặng weight_kg vừa trích xuất/impute
    result['output_urine_ml_kg_hr'] = result['output_urine_24h'] / (result['weight_kg'] * 24.0)

    # Cờ nguy cơ lâm sàng
    result['is_oliguria_24h'] = (result['output_urine_ml_kg_hr'] < 0.5).astype(int)    # KDIGO Stage 1
    result['is_anuria_24h'] = (result['output_urine_ml_kg_hr'] < 0.1).astype(int)      # Vô niệu/Thiểu niệu nặng
    result['is_fluid_overload_24h'] = (result['net_fluid_balance_24h'] > 2000.0).astype(int) # Quá tải dịch > 2 Lít

    # Đóng gói DataFrame cho Machine Learning
    df_ml = result.drop(columns=['intime', 'weight_kg'])

    return df_ml

In [72]:
urine_itemids = [
    226559,  # Foley catheter
    226560,  # Void (Tự tiểu)
    226561,  # Condom Cath
    226563,  # Straight Cath
    226564,  # Suprapubic catheter
    226565,  # R Nephrostomy (Dẫn lưu thận P)
    226567,  # L Nephrostomy (Dẫn lưu thận T)
    227488,  # GU Irrigant Out (Trừ dịch rửa bàng quang)
    227489   # TURP Irrigant Out
]

inputevents      = load("icu",  "inputevents.csv.gz")
outputevents     = load("icu",  "outputevents.csv.gz")

  ✓ icu/inputevents.csv.gz: 20,404 rows × 26 cols
  ✓ icu/outputevents.csv.gz: 9,362 rows × 9 cols


In [73]:
inputs_outputs = process_inputs_outputs_and_rag_text(inputevents, outputevents, train_df, urine_itemids=urine_itemids)

In [74]:
inputs_outputs.head()

,stay_id,input_total_volume_24h,output_total_volume_24h,output_urine_24h,net_fluid_balance_24h,output_urine_ml_kg_hr,is_oliguria_24h,is_anuria_24h,is_fluid_overload_24h
0,37067082,3097.250031,2745,2645,352.250031,1.574405,0,0,0
1,36753294,2533.553152,3660,3510,-1126.446848,2.089286,0,0,0
2,37049133,1444.166669,1227,427,217.166669,0.254167,1,0,0
3,35514836,6305.100023,2230,1079,4075.100023,0.642262,0,0,1
4,32128372,11213.600952,1875,185,9338.600952,0.110119,1,0,1


In [75]:
inputs_outputs.shape

(50, 9)

### **Nhánh 1: Sinh min, max, mean**

In [ ]:
from typing import List

def aggregate_tabular_features(
    cleaned_events_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    feature_list: List[str],
    prefix: str = "chart"
) -> pd.DataFrame:
    """
    Tạo các đặc trưng thống kê bảng (Nhánh 1): Min, Max, Mean, First, Last, Slope và Missing Indicator.
    """
    # Tạo bảng khung sẵn tất cả stay_id để đảm bảo không mất bệnh nhân
    result_df = pd.DataFrame({'stay_id': cohort_df['stay_id'].unique()})

    # Sắp xếp để tính First, Last, Slope chính xác
    df_sorted = cleaned_events_df.sort_values(by=['stay_id', 'feature_name', 'delta_hours'])

    for feat in feature_list:
        feat_data = df_sorted[df_sorted['feature_name'] == feat]

        if feat_data.empty:
            result_df[f"{prefix}_{feat}_is_missing"] = 1
            continue

        # Thống kê cơ bản
        agg_stats = feat_data.groupby('stay_id')['valuenum'].agg(
            min='min',
            max='max',
            mean='mean',
            first='first',
            last='last'
        ).reset_index()

        # Tính Slope (Tốc độ thay đổi theo giờ: (Last - First) / Delta_t)
        time_stats = feat_data.groupby('stay_id')['delta_hours'].agg(
            first_time='first',
            last_time='last'
        ).reset_index()

        merged_stats = agg_stats.merge(time_stats, on='stay_id')
        time_diff = merged_stats['last_time'] - merged_stats['first_time']

        # Nếu chỉ đo 1 lần hoặc khoảng cách thời gian = 0 thì Slope = 0
        merged_stats['slope'] = np.where(
            time_diff > 0,
            (merged_stats['last'] - merged_stats['first']) / time_diff,
            0.0
        )

        # Đổi tên cột theo prefix
        merged_stats = merged_stats.rename(columns={
            'min': f"{prefix}_{feat}_min",
            'max': f"{prefix}_{feat}_max",
            'mean': f"{prefix}_{feat}_mean",
            'first': f"{prefix}_{feat}_first",
            'last': f"{prefix}_{feat}_last",
            'slope': f"{prefix}_{feat}_slope"
        })[['stay_id', f"{prefix}_{feat}_min", f"{prefix}_{feat}_max",
            f"{prefix}_{feat}_mean", f"{prefix}_{feat}_first",
            f"{prefix}_{feat}_last", f"{prefix}_{feat}_slope"]]

        # Merge vào bảng kết quả chính
        result_df = result_df.merge(merged_stats, on='stay_id', how='left')

        # Thêm cờ Missing Indicator (1: Khuyết, 0: Có dữ liệu)
        result_df[f"{prefix}_{feat}_is_missing"] = result_df[f"{prefix}_{feat}_mean"].isna().astype(int)

    return result_df

In [ ]:
def serialize_events_to_narrative(
    cleaned_events_df: pd.DataFrame,
    stay_id: int
) -> str:
    """
    Chuyển đổi chuỗi diễn tiến sinh hiệu/xét nghiệm trong 24h thành văn bản chuỗi thời gian (Nhánh 2 - LLM).
    """
    patient_events = cleaned_events_df[cleaned_events_df['stay_id'] == stay_id].copy()

    if patient_events.empty:
        return "Không ghi nhận dữ liệu sinh hiệu/xét nghiệm nào trong 24 giờ đầu ICU."

    # Sắp xếp tăng dần theo mốc giờ
    patient_events = patient_events.sort_values(by='delta_hours')

    # Nhóm các chỉ số được đo cùng một mốc giờ (làm tròn đến giờ gần nhất)
    patient_events['hour_bucket'] = patient_events['delta_hours'].round().astype(int)
    grouped = patient_events.groupby('hour_bucket')

    narrative_lines = []
    for hour, group in grouped:
        measures = [f"{row['feature_name']}: {row['valuenum']:.1f}" for _, row in group.iterrows()]
        measures_str = ", ".join(measures)
        narrative_lines.append(f"- Giờ thứ {hour:02d}: {measures_str}")

    return "\n".join(narrative_lines)